# Forma HOOD Validation Notebook

Runs on **Kaggle free T4 GPU** (GPU accelerator must be enabled in Settings).

Validates the ContourCraft / HOOD simulation backend against the CPU XPBD solver by comparing `clearance_map` values per body region. Each region delta must be ≤ 2 mm.

## What this notebook does
1. Clone the Forma repo
2. Install ContourCraft and all its dependencies
3. Download pretrained ContourCraft weights
4. Run the **CPU solver** on `makehuman_male_M.ply` + `tshirt_size_M.json`
5. Run the **HOOD solver** on the same inputs
6. Compare `clearance_map` per region — flag any delta > 2 mm as FAIL

## Step 0 — Check GPU

In [ ]:
import subprocess, os, re
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,compute_cap', '--format=csv,noheader'],
                        capture_output=True, text=True)
print(result.stdout.strip())
assert result.returncode == 0, 'GPU not available — enable GPU accelerator in Kaggle Settings'

# Parse compute capability (e.g. '6.0' for P100, '7.5' for T4)
cc_match = re.search(r'(\d+)\.(\d+)\s*$', result.stdout.strip())
gpu_cc_major = int(cc_match.group(1)) if cc_match else 0
print(f'Compute capability: sm_{gpu_cc_major}x')

# PyTorch 2.10+ with CUDA 12.8 dropped sm_60 (P100) support.
# If we get a P100, use CPU for ContourCraft inference (works, just slower).
# DON'T downgrade torch — that breaks pytorch3d source builds.
if gpu_cc_major < 7:
    os.environ['HOOD_DEVICE'] = 'cpu'
    print(f'WARNING: P100 (sm_60) not supported by preinstalled PyTorch.')
    print('ContourCraft will run on CPU (slower but functional).')
else:
    os.environ['HOOD_DEVICE'] = 'cuda:0'
    print(f'GPU OK — ContourCraft will run on cuda:0')


## Step 1 — Clone Forma repo

In [ ]:
# Forma repo on GitHub
FORMA_REPO = 'https://github.com/cmpotter04-sys/forma.git'
FORMA_BRANCH = 'main'
FORMA_DIR = '/kaggle/working/Forma'

!git clone --depth 1 --branch {FORMA_BRANCH} {FORMA_REPO} {FORMA_DIR}
print('Forma cloned to', FORMA_DIR)

## Step 2 — Install ContourCraft and dependencies

In [ ]:
# ContourCraft repo
CC_DIR = '/kaggle/working/ContourCraft'
!git clone --depth 1 https://github.com/Dolorousrtur/ContourCraft.git {CC_DIR}

In [ ]:
# Stub smplx to satisfy ContourCraft's top-level imports.
# smplx is non-commercial licensed — we cannot install it.
# ContourCraft uses smplx only in SMPL body generation / LBS utilities,
# not in add_coarse_edges or inference code which is all we call.
import sys, types
import torch

def _noop(*a, **kw): return None

_smplx = types.ModuleType('smplx')

# smplx.lbs — all symbols imported by utils/lbs.py and mesh_creation.py
_lbs = types.ModuleType('smplx.lbs')
_lbs.blend_shapes = _noop
_lbs.vertices2joints = _noop
_lbs.batch_rodrigues = _noop
_lbs.batch_rigid_transform = _noop
_lbs.lbs = _noop
_smplx.lbs = _lbs

# smplx.utils — Tensor type alias
_utils = types.ModuleType('smplx.utils')
_utils.Tensor = torch.Tensor
_smplx.utils = _utils

# smplx.body_models — needed by datasets/from_any_pose.py etc.
_smplx.body_models = types.ModuleType('smplx.body_models')

# Fake SMPL / SMPLX model classes.
# Imported as: from smplx import SMPL, SMPLX
# Used only in dataset construction code, never during inference.
class _FakeSMPL:
    def __init__(self, *a, **kw): pass
    def __call__(self, *a, **kw):
        B = (a[0].shape[0] if a else 1) if a else 1
        out = type('SMPLOutput', (), {
            'vertices': torch.zeros(B, 6890, 3),
            'joints':   torch.zeros(B, 24, 3),
        })()
        return out
    def eval(self): return self
    def to(self, *a, **kw): return self

class _FakeSMPLX(_FakeSMPL):
    def __call__(self, *a, **kw):
        B = (a[0].shape[0] if a else 1) if a else 1
        out = type('SMPLXOutput', (), {
            'vertices': torch.zeros(B, 10475, 3),
            'joints':   torch.zeros(B, 55, 3),
        })()
        return out

_smplx.SMPL  = _FakeSMPL
_smplx.SMPLX = _FakeSMPLX
_smplx.create = lambda *a, **kw: _FakeSMPL()
_smplx.body_models.SMPL  = _FakeSMPL
_smplx.body_models.SMPLX = _FakeSMPLX

sys.modules['smplx'] = _smplx
sys.modules['smplx.lbs'] = _lbs
sys.modules['smplx.utils'] = _utils
sys.modules['smplx.body_models'] = _smplx.body_models
print('smplx stub installed (lbs + utils + body_models + SMPL + SMPLX)')

In [ ]:
# Stub libraries that ContourCraft's utils/icontour.py imports at module level
# but are NOT needed for inference with the aux/from_any_pose config.
#
# Stubbed modules:
#   cudf        - RAPIDS GPU dataframe
#   cugraph     - RAPIDS GPU graph analytics
#   cccollisions - ContourCraft CUDA collision solver
import sys, types
import numpy as np
import torch

def _noop_stub(*a, **kw): return None

# --- cudf ---
try:
    import cudf
except ImportError:
    _cudf = types.ModuleType('cudf')
    _cudf.DataFrame = type('DataFrame', (), {
        '__init__': lambda self, *a, **kw: None,
        '__getitem__': lambda self, k: np.array([]),
        'values': property(lambda self: np.array([])),
    })
    _cudf.Series = type('Series', (), {'__init__': lambda self, *a, **kw: None})
    _cudf.from_dlpack = _noop_stub
    sys.modules['cudf'] = _cudf

# --- cugraph ---
try:
    import cugraph
except ImportError:
    _cugraph = types.ModuleType('cugraph')
    _cugraph.connected_components = _noop_stub
    _cugraph.Graph = type('Graph', (), {'__init__': lambda self, *a, **kw: None})
    sys.modules['cugraph'] = _cugraph
    sys.modules['cugraph.experimental'] = types.ModuleType('cugraph.experimental')

# --- cccollisions (BVH + collision detection stubs) ---
try:
    import cccollisions
    if not hasattr(cccollisions, 'bvh'):
        raise ImportError('Empty module')
except ImportError:
    _cccol = types.ModuleType('cccollisions')
    _cccol.compute_collisions = _noop_stub
    _cccol.compute_self_collisions = _noop_stub
    _cccol.CCCollisions = type('CCCollisions', (), {
        '__init__': lambda self, *a, **kw: None,
        'forward': _noop_stub,
        '__call__': _noop_stub,
    })
    def _bvh_stub(triangles):
        B = triangles.shape[0]
        F = triangles.shape[1]
        device = triangles.device
        N = max(2 * F - 1, 1)
        bboxes = torch.zeros(B, N, 6, device=device)
        tree = torch.zeros(B, N, 5, dtype=torch.long, device=device)
        return bboxes, tree
    def _find_collisions_stub(bboxes, tree, triangles, threshold, max_pairs):
        B = triangles.shape[0]
        device = triangles.device
        return torch.full((B, max_pairs, 2), -1, dtype=torch.long, device=device)
    def _find_triangle_edge_candidates2_stub(bboxes, tree, triangles, max_collisions=64):
        B = triangles.shape[0]
        device = triangles.device
        collision_tensor = torch.full((B, max_collisions, 3), -1, dtype=torch.long, device=device)
        binmasks = torch.zeros((B, max_collisions), dtype=torch.bool, device=device)
        return collision_tensor, binmasks
    _cccol.bvh = _bvh_stub
    _cccol.find_collisions = _find_collisions_stub
    _cccol.find_triangle_edge_candidates2 = _find_triangle_edge_candidates2_stub
    sys.modules['cccollisions'] = _cccol
    print('cccollisions stubs active')

print('Inference stubs setup complete.')


In [ ]:
# Patch ContourCraft for Python 3.12 compatibility
# Python 3.12 forbids mutable defaults in dataclasses.
# Walk all .py files in the ContourCraft repo and apply the fix.
import re, os

def _patch_py312_dataclass_defaults(path):
    with open(path) as f:
        src = f.read()
    original = src

    # Ensure 'field' is imported from dataclasses
    if 'from dataclasses import' in src:
        src = re.sub(
            r'from dataclasses import ([^\n]+)',
            lambda m: ('from dataclasses import ' + m.group(1)
                       if 'field' in m.group(1).split(',')
                       else 'from dataclasses import field, ' + m.group(1)),
            src, count=1
        )
    elif 'dataclass' in src:
        src = 'from dataclasses import field\n' + src

    # Replace:  attr: SomeClass = SomeClass()
    # With:     attr: SomeClass = field(default_factory=SomeClass)
    src = re.sub(
        r'^(\s+)(\w+)\s*:\s*(\w+)\s*=\s*\3\(\s*\)\s*$',
        r'\1\2: \3 = field(default_factory=\3)',
        src, flags=re.MULTILINE
    )

    if src != original:
        with open(path, 'w') as f:
            f.write(src)
        return True
    return False

patched = []
for root, dirs, files in os.walk(CC_DIR):
    dirs[:] = [d for d in dirs if not d.startswith('.')]
    for fname in files:
        if fname.endswith('.py'):
            fpath = os.path.join(root, fname)
            if _patch_py312_dataclass_defaults(fpath):
                patched.append(os.path.relpath(fpath, CC_DIR))

if patched:
    print(f'Patched {len(patched)} files for Python 3.12 dataclass compatibility:')
    for p in patched:
        print(f'  {p}')
else:
    print('No mutable defaults found (already compatible)')


In [ ]:
import torch
torch_ver = torch.__version__.split('+')[0]          # e.g. '2.10.0'
cuda_ver  = torch.version.cuda.replace('.', '')      # e.g. '128'
print(f'torch={torch_ver}  cuda={cuda_ver}')

PYG_WHL = f'https://data.pyg.org/whl/torch-{torch_ver}+cu{cuda_ver}.html'
print(f'PyG wheel URL: {PYG_WHL}')


In [ ]:
# Install torch-geometric + sparse dependencies
# Note: pyg_library was merged into torch-geometric 2.x and is no longer a separate package
!pip install -q torch-geometric==2.4.0
!pip install -q torch_scatter torch_sparse torch_cluster torch_spline_conv -f {PYG_WHL}


In [ ]:
# Fallback: stub torch_scatter + torch_sparse if the pip install did not succeed.
# torch_scatter is used in utils/icontour.py for the icontour *training* loss.
# The icontour loss is NOT evaluated during inference (runner.valid_rollout()).
# Stubbing is therefore safe for Forma validation purposes.
import sys

def _install_scatter_stubs():
    import types, torch
    def _scatter(src, index, dim=0, out=None, dim_size=None, fill_value=0, reduce="add", **kw):
        size = dim_size if dim_size is not None else int(index.max().item()) + 1
        shape = list(src.shape)
        shape[dim] = size
        return torch.full(shape, fill_value, dtype=src.dtype, device=src.device)
    _tscatter = types.ModuleType("torch_scatter")
    _tscatter.scatter       = _scatter
    _tscatter.scatter_add   = _scatter
    _tscatter.scatter_mean  = _scatter
    _tscatter.scatter_mul   = _scatter
    _tscatter.scatter_max   = lambda src, index, *a, **kw: (_scatter(src, index, *a, **kw), index)
    _tscatter.scatter_min   = _tscatter.scatter_max
    sys.modules["torch_scatter"] = _tscatter
    # torch_sparse stub (torch_geometric may import it)
    _tsparse = types.ModuleType("torch_sparse")
    _tsparse.SparseTensor = type("SparseTensor", (), {
        "__init__": lambda self, *a, **kw: None,
    })
    sys.modules["torch_sparse"] = _tsparse
    sys.modules["torch_sparse.storage"] = types.ModuleType("torch_sparse.storage")
    print("torch_scatter + torch_sparse: fallback stubs installed")

try:
    import torch_scatter
    print(f"torch_scatter available")
except (ImportError, OSError):
    _install_scatter_stubs()


In [ ]:
# Install pytorch3d — build from source (~5 min)
import os
os.environ['FORCE_CUDA'] = '1'
os.environ['TORCH_CUDA_ARCH_LIST'] = '6.0;7.0;7.5;8.0'
!pip install -q 'git+https://github.com/facebookresearch/pytorch3d.git@stable'


In [ ]:
# Install CCCollision (ContourCraft's CUDA collision solver)
!git clone -q https://github.com/NVIDIA/cuda-samples.git /tmp/cuda-samples
import os; os.environ['CUDA_SAMPLES_INC'] = '/tmp/cuda-samples/Common'
!git clone -q https://github.com/Dolorousrtur/CCCollisions.git /tmp/CCCollisions
!pip install -q /tmp/CCCollisions

In [ ]:
# Remaining ContourCraft dependencies
# warp-lang is required by ContourCraft utils/warp_u/* AND by Forma Warp backend
!pip install -q warp-lang
!pip install -q omegaconf einops networkx munch trimesh scipy scikit-learn loguru


In [ ]:
# Forma dependencies (CPU solver + pattern loading)
!pip install -q trimesh scipy numpy ezdxf shapely

## Step 3 — Download ContourCraft pretrained weights

Download `contourcraft.pth` from the [Google Drive link](https://drive.google.com/file/d/1NfxAeaC2va8TWMjiO_gbAcVPnZ8BYFPD/view) and upload it to Kaggle as a dataset, or use `gdown` if the file is publicly accessible.

In [ ]:
import os, zipfile

# The Google Drive file '1NfxAeaC2va8TWMjiO_gbAcVPnZ8BYFPD' is the full
# ContourCraft data bundle (ccraft_data.zip). It extracts to:
#   ccraft_data/aux_data/       — SMPL aux data, garment dicts, datasplits
#   ccraft_data/examples/       — example pose sequences
#   ccraft_data/trained_models/ — pretrained model weights (contourcraft.pth)
# DO NOT treat the bundle ZIP itself as the checkpoint — it is not loadable by torch.load.
GDRIVE_FILE_ID  = '1NfxAeaC2va8TWMjiO_gbAcVPnZ8BYFPD'
CCRAFT_DATA_ZIP = '/kaggle/working/ccraft_data.zip'
CCRAFT_DATA_DIR = '/kaggle/working/ccraft_data'
CHECKPOINT_PATH = f'{CCRAFT_DATA_DIR}/trained_models/contourcraft.pth'

if not os.path.exists(CHECKPOINT_PATH):
    # Download the bundle
    if not os.path.exists(CCRAFT_DATA_ZIP):
        !pip install -q gdown
        !gdown {GDRIVE_FILE_ID} -O {CCRAFT_DATA_ZIP}
    # Extract — creates /kaggle/working/ccraft_data/
    print(f'Extracting {CCRAFT_DATA_ZIP} ...')
    with zipfile.ZipFile(CCRAFT_DATA_ZIP, 'r') as zf:
        zf.extractall('/kaggle/working/')
    print('Extraction complete.')

assert os.path.exists(CHECKPOINT_PATH), (
    f'Checkpoint not found at {CHECKPOINT_PATH}. '
    'Ensure ccraft_data.zip was downloaded and extracted correctly.'
)
print(f'Checkpoint: {CHECKPOINT_PATH}  ({os.path.getsize(CHECKPOINT_PATH) / 1024 / 1024:.1f} MB)')
print(f'Data root:  {CCRAFT_DATA_DIR}')

In [ ]:
# Patch checkpoint: PyTorch 2.x requires two records in the checkpoint ZIP:
#   {prefix}/version  — protocol version
#   {prefix}/data.pkl — pickled state dict
# Old ContourCraft checkpoints (ccraft_data prefix) are missing both:
# the pickle is named '{prefix}/{prefix}.pkl' not '{prefix}/data.pkl'.
# Detect and patch both in one pass.
import zipfile

def _fix_checkpoint(path):
    try:
        with zipfile.ZipFile(path, 'r') as zf:
            names = zf.namelist()
        prefixes = set(n.split('/')[0] for n in names if '/' in n)
        prefix = next(iter(prefixes)) if len(prefixes) == 1 else 'archive'
        print(f'Checkpoint archive prefix: {prefix!r}')
        print(f'ZIP entries: {names[:10]}{'...' if len(names) > 10 else ''}')

        to_add = []

        version_key = f'{prefix}/version'
        if version_key not in names:
            to_add.append((version_key, '3'))

        data_pkl_key = f'{prefix}/data.pkl'
        if data_pkl_key not in names:
            pkl_candidates = [n for n in names if n.endswith('.pkl') and n.startswith(prefix + '/')]
            if pkl_candidates:
                print(f'Found non-standard pickle: {pkl_candidates[0]!r} -> aliasing to {data_pkl_key!r}')
                with zipfile.ZipFile(path, 'r') as zf:
                    pkl_bytes = zf.read(pkl_candidates[0])
                to_add.append((data_pkl_key, pkl_bytes))
            else:
                print(f'WARNING: no .pkl found under {prefix!r} — torch.load may fail')

        if to_add:
            with zipfile.ZipFile(path, 'a') as zf:
                for key, content in to_add:
                    zf.writestr(key, content)
            print(f'Checkpoint patched: added {[k for k, _ in to_add]}')
        else:
            print('Checkpoint OK — no patch needed')
    except Exception as e:
        print(f'WARNING: Could not patch checkpoint: {e}')

_fix_checkpoint(CHECKPOINT_PATH)


## Step 4 — Configure paths and run CPU solver

In [ ]:
import sys

FORMA_SRC = f'{FORMA_DIR}/src'
if FORMA_SRC not in sys.path:
    sys.path.insert(0, FORMA_SRC)

# Paths to test assets
BODY_MESH     = f'{FORMA_DIR}/data/bodies/makehuman_male_M.ply'
PATTERN       = f'{FORMA_DIR}/data/patterns/tshirt_size_M.json'
SEAM_MANIFEST = f'{FORMA_DIR}/data/seam_manifests/tshirt_size_M_seam_manifest.json'
FABRIC_ID     = 'cotton_jersey_default'

# Verify test assets exist
for p in [BODY_MESH, PATTERN, SEAM_MANIFEST]:
    assert os.path.exists(p), f'Missing test asset: {p}'
print('All test assets found.')

In [ ]:
import time
from pipeline import run_fit_check

print('Running CPU solver...')
t0 = time.perf_counter()
cpu_verdict = run_fit_check(
    body_mesh_path=BODY_MESH,
    pattern_path=PATTERN,
    seam_manifest_path=SEAM_MANIFEST,
    fabric_id=FABRIC_ID,
    backend='cpu',
)
cpu_ms = int((time.perf_counter() - t0) * 1000)
print(f'CPU solver done in {cpu_ms} ms')

cpu_clearance = {r['region']: r['delta_mm'] for r in cpu_verdict['strain_map']}
print('\nCPU clearance_map:')
for region, delta in sorted(cpu_clearance.items()):
    print(f'  {region:20s}: {delta:+.1f} mm')

## Step 5 — Run HOOD solver

In [ ]:
import os
os.environ['CONTOURCRAFT_ROOT'] = CC_DIR
os.environ['CONTOURCRAFT_CHECKPOINT'] = CHECKPOINT_PATH

# Set ContourCraft defaults so its internal path resolution works
cc_root_inserted = CC_DIR not in sys.path
if cc_root_inserted:
    sys.path.insert(0, CC_DIR)

from utils.defaults import DEFAULTS
DEFAULTS.project_dir = CC_DIR
DEFAULTS.data_root   = CCRAFT_DATA_DIR  # ccraft_data/ bundle with aux_data, trained_models, etc.

print('Running HOOD solver...')
t0 = time.perf_counter()
hood_verdict = run_fit_check(
    body_mesh_path=BODY_MESH,
    pattern_path=PATTERN,
    seam_manifest_path=SEAM_MANIFEST,
    fabric_id=FABRIC_ID,
    backend='hood',
)
hood_ms = int((time.perf_counter() - t0) * 1000)
print(f'HOOD solver done in {hood_ms} ms')

hood_clearance = {r['region']: r['delta_mm'] for r in hood_verdict['strain_map']}
print('\nHOOD clearance_map:')
for region, delta in sorted(hood_clearance.items()):
    print(f'  {region:20s}: {delta:+.1f} mm')

## Step 6 — Compare results (pass if all region deltas ≤ 2 mm)

In [ ]:
TOLERANCE_MM = 2.0

print(f'{'Region':<22} {'CPU (mm)':>10} {'HOOD (mm)':>10} {'|diff| (mm)':>12} {'Result':>8}')
print('-' * 66)

all_pass = True
for region in sorted(cpu_clearance.keys()):
    cpu_val  = cpu_clearance[region]
    hood_val = hood_clearance.get(region, float('nan'))
    diff     = abs(cpu_val - hood_val)
    passed   = diff <= TOLERANCE_MM
    status   = 'PASS' if passed else 'FAIL'
    if not passed:
        all_pass = False
    print(f'{region:<22} {cpu_val:>+10.2f} {hood_val:>+10.2f} {diff:>12.2f} {status:>8}')

print('-' * 66)
print()

if all_pass:
    print(f'OVERALL: PASS — all regions within {TOLERANCE_MM} mm tolerance')
else:
    print(f'OVERALL: FAIL — one or more regions exceed {TOLERANCE_MM} mm tolerance')
    raise AssertionError(
        f'HOOD vs CPU clearance delta exceeds {TOLERANCE_MM} mm tolerance on one or more regions.'
    )

## Summary

In [ ]:
print('=== Validation Summary ===')
print(f'CPU solver:  {cpu_ms} ms   fit={cpu_verdict["fit"]}')
print(f'HOOD solver: {hood_ms} ms  fit={hood_verdict["fit"]}')
print(f'Tolerance:   {TOLERANCE_MM} mm per region')
print(f'Result:      {"PASS" if all_pass else "FAIL"}')